# L09 · 有方向的量 — 向量代数

**目标**：明白向量就是一串数，能加、能缩放，还有几何含义。

**为什么对 Aurora 重要**：Aurora 的 MFCC 特征提取把每帧音频压缩成一个 `(13,)` 向量；`scale(v, c)` 调音量时就是对这个向量做标量缩放。神经网络每层的激活值同样是向量。

## 本课剧情：让一串数变成箭头

向量的两个基本操作——加法和标量缩放——是整个线代的地基：矩阵、特征分解、SVD 都建立在这两个运算上。本节先用 `[3.0, 4.0]` 这样的小向量把这两个操作走通，建立 shape、长度和方向的直觉。

## 1. 向量就是 numpy 数组

在线性代数里，向量是一个有序数列 `[x₁, x₂, …, xₙ]`，NumPy 用一维 array 表示，shape 为 `(n,)`。两个必检属性：`.shape` 告诉你维度，`.dtype` 告诉你数值类型（默认 `float64`）。

为什么不用 Python list？NumPy array 支持向量化运算——`c * v` 直接对所有元素并行缩放，速度比 Python 循环快几十倍，且语法与数学公式一一对应。Aurora 的 `audio_core/mfcc.py` 里，`extract_mfcc()` 把每帧 25ms 音频压缩成一个 `(13,)` float64 array，正是这里所说的向量——维度 = 特征数，每个元素 = 一个梅尔倒谱系数。

## 符号入口：先看形状，再看运算

线性代数里的对象都有明确形状：向量是 `(n,)`，矩阵是 `(m, n)`，矩阵乘向量会把 `(n,)` 变成 `(m,)`。每个例子都先标出输入、变换和输出。

In [ ]:
import numpy as np
v = np.array([3.0, 4.0])      # 2 维向量
audio = np.array([0.0, 0.5, 1.0, 0.5, 0.0])  # 5 个采样点 = 5 维向量
print('v =', v, '| 维度:', v.shape)
print('audio 是一个', audio.shape[0], '维向量')

## 动手观察：shape 和 norm

观察 `v.shape` 输出 `(2,)`、`A.shape` 输出 `(2, 2)`、`A @ v` 的结果——矩阵把向量拉伸后 shape 不变但值变了。`np.linalg.norm(v)` 返回 5.0，即 √(3²+4²)，这是向量长度的 NumPy 写法。

In [ ]:
import numpy as np

v = np.array([3.0, 4.0])
A = np.array([[2.0, 0.0],
              [0.0, 0.5]])

print('v =', v, 'shape =', v.shape)
print('A =')
print(A)
print('A shape =', A.shape)
print('A @ v =', A @ v)
print('向量长度 ||v|| =', np.linalg.norm(v))


## 代码实验：遍历几个向量，观察矩阵如何改变它们

把四个方向各异的向量喂给同一个矩阵 `A`，看 x 分量如何被 y 分量"拉偏"——非对角元素 `A[0][1]=1.0` 就是这个耦合的来源。

In [ ]:
import numpy as np

A = np.array([[2.0, 1.0],
              [0.0, 1.0]])
vectors = [np.array([1.0, 0.0]), np.array([0.0, 1.0]), np.array([1.0, 1.0]), np.array([2.0, -1.0])]

print('A =')
print(A)
for v in vectors:
    out = A @ v
    print(f'v={v} -> A@v={out}')


## 2. 加法 = 逐元素相加；缩放 = 每个元素乘同一个数

In [ ]:
a = np.array([1.0, 2.0])
b = np.array([3.0, 1.0])
print('a + b =', a + b)     # 信号叠加（混音）就是向量加法
print('2 * a =', 2 * a)     # 放大 2 倍 = 标量缩放

## 3. ✏️ 你的任务：实现 `scale`

`scale(v, c)` 把向量 `v` 的每个分量都乘以标量 `c`——这是音量旋钮的数学本质。

**推理路线**：
1. 标量缩放的几何意义：向量方向不变，长度变为原来的 `|c|` 倍；当 `c < 0` 时方向反转，`c = 0` 时退化为零向量。
2. NumPy 广播：`c * v` 把标量 `c` 自动应用到 `v` 的每个元素——无需写 for 循环，底层是 SIMD 并行指令。
3. 返回 shape：`c * v` 的 shape 与 `v` 完全相同，不需要额外 reshape。

**参考输入输出**：`scale([1.0, 2.0], 2.0)` → `[2.0, 4.0]`；`scale([0.2, -0.4], 1.5)` → `[0.3, -0.6]`

<details><summary>点击查看参考实现</summary>

```python
def scale(v, c):
    return c * v
```

</details>

### 写 `scale` 前明确三件事

- 输入：`v`（numpy array，任意 shape）；`c`（浮点数，缩放系数）
- 关键步骤：`c * v`，NumPy 的标量乘法对每个元素同等作用
- 返回：与 `v` 等 shape 的 array，每个元素值为原来的 `c` 倍

In [ ]:
def scale(v, c):
    # ✏️ TODO: 返回 c 倍的 v
    return None  # ← 改这里


In [ ]:
audio = np.array([0.2, -0.4, 0.6, -0.8])
louder = scale(audio, 1.5)
print('原始:', audio)
print('放大:', louder)
assert np.allclose(louder, [0.3, -0.6, 0.9, -1.2]), '应是每个元素 ×1.5'
print('\n✅ 通过：你会缩放向量了 = 调音量。')

## 4. 几何：向量是带方向的箭头

In [ ]:
import matplotlib.pyplot as plt
v = np.array([3.0, 4.0])
plt.figure(figsize=(4,4))
plt.quiver(0,0, v[0], v[1], angles='xy', scale_units='xy', scale=1)
plt.xlim(-1,5); plt.ylim(-1,6); plt.grid(True, alpha=.3)
plt.gca().set_aspect('equal'); plt.title('vector [3,4]'); plt.show()

**🔗 Aurora 连接**：`audio_core/mfcc.py` 的 `extract_mfcc()` 输出每帧 `(13,)` 向量；`scale(frame, gain)` 对这个向量做逐元素缩放，直接用于 Aurora 的音量归一化步骤（调音量 = 乘标量，不改变频谱形状）。下一课点积会让你定量比较两帧 MFCC 向量的相似度——语音检索和注意力机制的底层操作。

## 🎨 图示：向量 = 带方向的箭头，缩放 = 改变长度

In [ ]:
from laviz import style, arrows2d
style()
arrows2d([[3,4],[1,2],[6,8]], ['a=[3,4]','b=[1,2]','2a (缩放)'],
         title='向量与缩放');

In [ ]:
A = np.array([[2.0, 1.0], [0.0, 1.0]])
probes = np.array([[1,0], [0,1], [1,1], [-1,2]], dtype=float)
print('矩阵 A 会怎样移动这些向量？')
for v in probes:
    out = A @ v
    print(f'{v} -> {out} | 长度 {np.linalg.norm(v):.2f} -> {np.linalg.norm(out):.2f}')


## 参数实验：只改一个旋钮

1. 把 `A[0][1]` 从 `1.0` 改成 `0.0`——矩阵变成对角矩阵，x 分量不再受 y 影响；验证非对角元素是"分量耦合"的唯一来源。
2. 把 `A[0][1]` 改成负数（如 `-2.0`），先手算 `v=[1, 1]` 的输出，再运行对照——负值会把 x 分量"拉向反方向"。
3. 把 `probes` 里一个向量换成 `[3.0, 4.0]`（norm = 5.0），观察矩阵对"长"向量的缩放倍率是否与对 `[1, 1]` 相同（提示：矩阵变换一般**不**保持 norm 不变，除非是正交矩阵）。

In [ ]:
A = np.array([[2.0, 1.0], [0.0, 1.0]])
probes = np.array([[1,0], [0,1], [1,1], [-1,2]], dtype=float)
print('矩阵 A 会怎样移动这些向量？')
for v in probes:
    out = A @ v
    print(f'{v} -> {out} | 长度 {np.linalg.norm(v):.2f} -> {np.linalg.norm(out):.2f}')


## 本课收束

现在可以用 `scale(v, c)` 对任意 numpy array 做逐元素缩放，输出 shape 与输入完全相同。这对应 Aurora `audio_core` 里的调音量操作，而向量加法对应混音——两个操作合起来覆盖了信号合成的底层逻辑。MFCC 特征提取最终把每帧压成一个 `(13,)` 向量，就是今天操作的对象形式。下一节点积会让你计算两个向量的相似度，这是特征匹配和注意力机制的基础运算。

In [ ]:
# 小检查：先看 shape，再做运算
import numpy as np

a = np.array([1.0, 2.0])
b = np.array([3.0, 4.0])
A = np.array([[1.0, 2.0], [0.0, 1.0]])

print('a.shape =', a.shape)
print('b.shape =', b.shape)
print('A.shape =', A.shape)
print('a + b =', a + b)
print('A @ a =', A @ a)
print('a dot b =', float(a @ b))
